# 02 — Model 1: Custom CNN Training (From Scratch)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arudaev/chexvision/blob/main/notebooks/02_train_scratch.ipynb)

Train the custom ResNet-style CNN from scratch on NIH Chest X-ray14.

> **Cloud-only**: This notebook MUST run on Google Colab (free T4 GPU) or a university cluster. Do NOT run locally.

**Architecture decisions documented here:**
- Residual blocks (skip connections for gradient flow)
- Batch normalization (training stability)
- Global average pooling (parameter efficiency)
- Kaiming initialization (suited for ReLU)
- AdamW optimizer (decoupled weight decay)
- Cosine annealing LR (smooth decay without sharp drops)

In [ ]:
# === Colab Setup ===
import os, subprocess, sys

# Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected! Go to Runtime > Change runtime type > T4 GPU. '
        'Training 112k images on CPU is not feasible.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Clone repo and install
if not os.path.exists('/content/chexvision'):
    subprocess.run(['git', 'clone', 'https://github.com/arudaev/chexvision.git', '/content/chexvision'], check=True)
%cd /content/chexvision
!pip install -q -e '.[dev]'
os.environ['CHEXVISION_ALLOW_LOCAL'] = '1'  # We're in Colab, override the guard

In [ ]:
# Download dataset to Colab's ephemeral storage
!python -m src.data.download --output-dir /content/data
print('Dataset ready at /content/data')

In [ ]:
import yaml
from src.models.scratch_cnn import CheXVisionScratch

# Load config
with open('configs/scratch.yaml') as f:
    config = yaml.safe_load(f)

# Point to Colab data directory
config['data']['data_dir'] = '/content/data'
config['logging'] = {'checkpoint_dir': '/content/checkpoints', 'log_dir': '/content/logs'}

# Model architecture summary
model = CheXVisionScratch()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'\nConfig:\n{yaml.dump(config, default_flow_style=False)}')

In [ ]:
# Train
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from src.training.trainer import train
train(config)

In [ ]:
# Upload trained model to HuggingFace Hub
from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get('HF_TOKEN'))  # Set in Colab Secrets
api.upload_file(
    path_or_fileobj='/content/checkpoints/CheXVision-ResNet_best.pth',
    path_in_repo='CheXVision-ResNet_best.pth',
    repo_id='HlexNC/chexvision-scratch',
    repo_type='model',
)
print('Model uploaded to https://huggingface.co/HlexNC/chexvision-scratch')